# Probar pronunciaciones

Para cuando el narrador pronuncia mal una palabra: prueba varias escrituras con **tu voz y la configuración calibrada** (`config/voz.yaml`), las compara con Whisper y te deja escucharlas. La que elijas se añade a `config/pronunciacion.yaml`; la próxima ejecución de la etapa 2 regenera sola las frases afectadas.

Caso típico: palabras que también existen en inglés (*red, final, normal, real, total, general…*). El modelo multilingüe comparte vocabulario entre idiomas y a veces las lee con acento inglés. Escribirlas de forma que no sean una palabra inglesa (`rred`, `réd`) suele bastar.

Whisper solo confirma que se entiende; **el acento lo juzga tu oído**. Kernel: `series-redes-neuronales (.venv 3.13)`. Tarda ~1 min por cada 10 audios.

In [ ]:
import gc, itertools, sys
from pathlib import Path

RAIZ = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "pipeline" / "produce.py").exists()), None)
if RAIZ is None:
    raise RuntimeError("Abre el notebook desde dentro de series-redes-neuronales/")
if Path(sys.prefix).resolve() != (RAIZ / ".venv").resolve():
    raise RuntimeError(f"Kernel equivocado: {sys.executable}. "
                       f"Selecciona 'series-redes-neuronales (.venv 3.13)' o {RAIZ / '.venv/bin/python'}")
sys.path.insert(0, str(RAIZ))

import jiwer
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, HTML, display

from lib import audio, config
from lib.texto import erre_inicial, numeros_a_palabras, normalizar_para_wer
from pipeline.e2_tts import Oido, Voz, limpiar_bordes   # mismos modelos y limpieza que el pipeline

print(RAIZ, "·", torch.__version__, "·", torch.cuda.get_device_name(0))

## Qué probar

Cada caso es una palabra, una frase donde aparece (con `{}` en su lugar) y las escrituras candidatas; la primera variante es la palabra tal cual, como referencia.

El caso `regla_rr` compara una frase entera **sin** y **con** la regla `erre_inicial` (r inicial → rr en todas las palabras). Si la versión con regla suena mejor en todas las tomas, actívala en `config/voz.yaml` (`erre_inicial: true`) y sobra añadir palabras con r una a una.

In [ ]:
FRASE_R = "La red recibe el resultado, lo redondea rápidamente y reduce el ruido."

# Parámetros a comparar. cfg_weight manda cuánto se ciñe el modelo al texto y al idioma frente a
# imitar el audio de referencia: si se cuela acento inglés en palabras como "red", subirlo suele ayudar.
PARAMETROS = {
    "calibrada": None,                  # la de config/voz.yaml (ahora C: cfg 0.3)
    "cfg 0.5": {"cfg_weight": 0.5},
    "cfg 0.7": {"cfg_weight": 0.7},
}

CASOS = {
    "regla_rr": {   # textos completos: sin regla y con regla
        "referencia": FRASE_R,
        "textos": {"sin regla": FRASE_R, "con regla": erre_inicial(FRASE_R, respetar_diccionario=False)},
        "parametros": PARAMETROS,
    },
    # Ya resuelto (red → "rred" en el diccionario). Se deja como ejemplo de caso por palabra:
    # "red": {"frase": "Y sin embargo, una {} neuronal encontrará la frontera ella sola.",
    #         "variantes": ["red", "rred", "réd", "rréd"]},
    # Otras palabras que existen en inglés y aparecerán a menudo en la serie. Descomenta para probarlas.
    # "final":  {"frase": "La capa {} decide la clase.",          "variantes": ["final", "finál"]},
    # "normal": {"frase": "Una distribución {} con media cero.",  "variantes": ["normal", "normál"]},
}
SEMILLAS = [42, 1042]   # varias tomas por variante: una sola puede engañar
SALIDA = RAIZ / "build/pronunciacion"

## Generar y transcribir

In [ ]:
voz_cfg = config.cargar("voz")
voz = Voz(voz_cfg)   # los parámetros se sustituyen por caso más abajo
oido = Oido(config.cargar("pipeline")["voz_qa"]["modelo_whisper"])

filas = []
for palabra, caso in CASOS.items():
    if "textos" in caso:
        referencia, textos = caso["referencia"], caso["textos"]
    else:
        referencia = caso["frase"].format(palabra)
        textos = {v: caso["frase"].format(v) for v in caso["variantes"]}
    for (variante, texto), (nombre_par, params) in itertools.product(textos.items(), caso.get("parametros", {"calibrada": None}).items()):
        texto = numeros_a_palabras(texto)
        voz.cfg = {**voz_cfg, "parametros": {**voz_cfg["parametros"], **(params or {})}}
        for semilla in SEMILLAS:
            ruta = SALIDA / palabra / f"{variante}_{nombre_par}_{semilla}.wav".replace(" ", "_")
            ruta.parent.mkdir(parents=True, exist_ok=True)
            wav, sr = voz.sintetizar(texto, semilla)
            wav = audio.normalizar_sonoridad(audio.recortar_silencios(wav, sr), sr)
            sf.write(ruta, wav, sr)
            transcripcion, palabras = oido.transcribir(ruta)
            wav, _, _ = limpiar_bordes(wav, sr, palabras)
            sf.write(ruta, wav, sr)
            ref, hip = normalizar_para_wer(referencia).split(), normalizar_para_wer(transcripcion).split()
            filas.append(dict(palabra=palabra, variante=f"{variante} · {nombre_par}", semilla=semilla, ruta=ruta,
                              wer=jiwer.wer(" ".join(ref), " ".join(hip)), transcripcion=transcripcion))
        print(f"{palabra} → {variante} listo")

res = pd.DataFrame(filas)
del voz, oido; gc.collect(); torch.cuda.empty_cache()
display(res.groupby(["palabra", "variante"], sort=False).wer.agg(["mean", "max", "count"]).round(2)
        .rename(columns={"mean": "WER medio", "max": "WER peor toma", "count": "tomas"}))

## Escuchar

Compara las variantes de cada palabra. Busca la que suene a español nativo en todas las tomas, no solo en una.

In [ ]:
for palabra in CASOS:
    display(HTML(f"<h3>{palabra}</h3>"))
    for variante, grupo in res[res.palabra == palabra].groupby("variante", sort=False):
        display(HTML(f"<h4>«{variante}»</h4>"))
        for fila in grupo.itertuples():
            print(f"semilla {fila.semilla}: WER {fila.wer:.2f} «{fila.transcripcion}»")
            display(Audio(str(fila.ruta)))

## Guardar la elegida

Escribe la variante elegida para cada palabra (o déjala fuera si ninguna mejora). Se añade al final de `config/pronunciacion.yaml`; si la palabra ya estaba, se sustituye.

In [ ]:
ELEGIDAS = {
    # "palabra": "variante",   ← por ejemplo "final": "finál"
}

ruta = RAIZ / "config/pronunciacion.yaml"
lineas = ruta.read_text(encoding="utf-8").splitlines()
for palabra, variante in ELEGIDAS.items():
    if variante == palabra:
        continue
    lineas = [l for l in lineas if not l.split("#")[0].strip().lower().startswith(f"{palabra.lower()}:")]
    lineas.append(f'{palabra}: "{variante}"   # acento inglés (notebooks/02_probar_pronunciacion)')
ruta.write_text("\n".join(lineas) + "\n", encoding="utf-8")
config.cargar.cache_clear()
print(ruta.read_text(encoding="utf-8"))